In [ ]:
import sys
import os
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import math

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
matplotlib.use('QtAgg')
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qtagg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
from matplotlib.animation import FuncAnimation
from mpl_toolkits.axes_grid1 import make_axes_locatable

from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, 
    QComboBox, QLabel, QSpinBox, QPushButton, QStackedWidget, QGroupBox,
    QTableWidget, QTableWidgetItem, QHeaderView, QCheckBox, QButtonGroup,
    QRadioButton, QLineEdit
)
from PyQt6.QtCore import Qt, QTimer

from animationGUI import AnimationData

CMTOMICRON = 1e4
VCMTOkVCM = 1e-3



In [ ]:
allData = AnimationData()

In [ ]:
allData.avalancheData

In [ ]:
plt.hist(allData.avalancheData['Gain'])
plt.show()

In [ ]:
allData.animationData

In [ ]:
pitch = allData.simData['pitch']
padLength = allData.simData['padLength']
radius = allData.simData['holeRadius']

fieldData = allData.fieldStrengths.copy()
fieldData['ratio'] = fieldData['Weight_RightBottomPad'] / fieldData['Weight_TopPad']

#plotData = np.log10(fieldData['ratio'])
plotData = fieldData['ratio']
mask = np.isclose(fieldData['x'], 0, atol=1e-6)

xData = fieldData['y'][mask].to_numpy()
yData = fieldData['z'][mask].to_numpy()
zData = plotData[mask].to_numpy()

# Filter out NaN and Inf values
valid = np.isfinite(zData)
xData, yData, zData = xData[valid], yData[valid], zData[valid]

contour = plt.tricontourf(
    xData, yData, zData, 
    levels=np.linspace(0, 1, 101), cmap='viridis', extend='both' 
)

cbar = plt.colorbar(contour)
cbar.set_ticks(np.linspace(0, 1, 11))
cbar.set_label('Field Ratio: Bottom / Center ', rotation=270, labelpad=15, fontsize=14)

lineLevels = [1, .5, .1, .01]
lineStyles = ['-', '--', '-.', ':']
for inLevel, inLine in zip(lineLevels, lineStyles):
    contourLine = plt.tricontour(
        xData, yData, zData,
        levels=[inLevel], 
        colors='c', 
        linestyles=inLine,
        linewidths=2.5
    )
    plt.clabel(contourLine, inline=True, fontsize=14, fmt=f'{inLevel}')
    plt.plot([], [], c='c', ls=inLine, lw=2.5, label=f'{inLevel}')

plt.plot([-pitch+radius, -radius], [0,0], c='k', lw=5)
plt.plot([radius, pitch-radius], [0,0], c='k', lw=5)

plt.plot([-pitch-padLength, -pitch+padLength], [-50,-50], c='m', lw=5)
plt.plot([-padLength, padLength], [-50,-50], c='m', lw=5)
plt.plot([pitch-padLength, pitch+padLength], [-50,-50], c='m', lw=5)

plt.xlabel(r'y ($\mu$m)')
plt.ylabel(r'z ($\mu$m)')
plt.xlim([-pitch, pitch])
plt.ylim([-50, 50])

plt.legend()
plt.show()